# One-dimensional illustrative Re-SC example

This notebook constructs a fixed synthetic binary dataset with class imbalance and a deliberately overlapping region. It applies the package implementations of Re-SC and KMeans-ReSC and creates a publication-ready comparison figure.

> **Interpretation boundary:** this is an illustrative example of the methods' geometric behavior. It is not experimental evidence that KMeans-ReSC always improves class separability.

## Imports and output location

Import the two resamplers and locate the paper's figure directory. The same random state is used for data generation and both algorithms.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from imblearn_resc.oversampling import KMeansReSC, ReSC

RANDOM_STATE = 42

project_root = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "latex").is_dir() and (path / "resc-experiments").is_dir()
)
figure_dir = project_root / "latex" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

print(f"Figures will be saved to: {figure_dir}")

## Generate the fixed one-dimensional dataset

The majority class contains 300 observations: 240 belong to a separated mode and 60 belong to a mode that overlaps the minority distribution. The minority class contains 100 observations, giving an initial imbalance ratio of 3. The distributions and seed are declared directly; no seed search is performed.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

majority_separated = rng.normal(loc=-2.0, scale=0.55, size=240)
majority_overlapping = rng.normal(loc=0.40, scale=0.45, size=60)
minority = rng.normal(loc=0.50, scale=0.45, size=100)

X = np.concatenate([majority_separated, majority_overlapping, minority]).reshape(-1, 1)
y = np.concatenate([
    np.zeros(300, dtype=int),
    np.ones(100, dtype=int),
])

print(f"X shape: {X.shape}")
print(f"Majority: {(y == 0).sum()}, minority: {(y == 1).sum()}, IR: {(y == 0).sum() / (y == 1).sum():.2f}")

## Apply Re-SC and KMeans-ReSC

Both methods use $M=1.5$ and the same fixed random state. KMeans-ReSC uses the paper configuration for the neighborhood filter and the bounded candidate grid. Because sample concatenation maps $d$ features to $2d$ features, the one-dimensional input becomes a two-dimensional resampled dataset.

In [ ]:
resc = ReSC(
    M=1.5,
    k=5,
    alpha=0.05,
    epsilon=0.05,
    random_state=RANDOM_STATE,
)

kmeans_resc = KMeansReSC(
    M=1.5,
    random_state=RANDOM_STATE,
    n_neighbors=5,
    safe_threshold=0.9,
    max_k_candidates=10,
    kmeans_params={"n_init": 10},
)

X_resc, y_resc = resc.fit_resample(X, y)
X_kmeans, y_kmeans = kmeans_resc.fit_resample(X, y)

print(f"Re-SC output shape: {X_resc.shape}")
print(f"KMeans-ReSC output shape: {X_kmeans.shape}")

## Inspect class balance and representative selection

Report the resulting class counts and algorithm diagnostics. Unique values in the second Re-SC coordinate recover the number of selected raw representatives for this one-dimensional example. KMeans-ReSC exposes its selected centroid count directly.

In [ ]:
def describe_resampled(name, X_resampled, y_resampled):
    labels, counts = np.unique(y_resampled, return_counts=True)
    count_by_label = dict(zip(labels, counts))
    majority_count = count_by_label[0]
    minority_count = count_by_label[1]
    ir = max(majority_count, minority_count) / min(majority_count, minority_count)
    print(
        f"{name}: majority={majority_count:,}, minority={minority_count:,}, "
        f"IR={ir:.3f}, dimensions={X_resampled.shape[1]}"
    )

describe_resampled("Re-SC", X_resc, y_resc)
describe_resampled("KMeans-ReSC", X_kmeans, y_kmeans)

resc_representative_count = np.unique(X_resc[y_resc == 0, 1]).size
print(f"Re-SC representatives: {resc_representative_count}")
print(f"KMeans-ReSC selected k: {kmeans_resc.selected_k_}")
print(f"KMeans-ReSC safe majority inputs: {kmeans_resc.safe_majority_count_} / 300")
print(f"KMeans-ReSC candidate k values: {kmeans_resc.candidate_ks_}")
print(f"Safety-filter fallback used: {kmeans_resc.fallback_used_}")
print(f"Candidate-selection fallback used: {kmeans_resc.selection_fallback_used_}")

## Plot the original and resampled datasets

The first panel shows the original one-dimensional class-count distributions. The remaining panels show the actual two-dimensional concatenated outputs on identical axes. Low-opacity points reveal density while rasterization keeps the PDF compact.

In [ ]:
majority_color = "#3568A8"
minority_color = "#D85745"

fig, axes = plt.subplots(1, 3, figsize=(10.4, 3.25), constrained_layout=True)

bins = np.linspace(X.min() - 0.15, X.max() + 0.15, 30)
axes[0].hist(
    X[y == 0, 0], bins=bins, color=majority_color, alpha=0.58,
    label="Majority ($n=300$)", edgecolor="white", linewidth=0.25,
)
axes[0].hist(
    X[y == 1, 0], bins=bins, color=minority_color, alpha=0.72,
    label="Minority ($n=100$)", edgecolor="white", linewidth=0.25,
)
axes[0].set_title("(a) Original 1D data", fontsize=10)
axes[0].set_xlabel("Feature value $x$")
axes[0].set_ylabel("Count")
axes[0].text(0.03, 0.95, "IR = 3.00", transform=axes[0].transAxes, va="top", fontsize=8.5)
axes[0].legend(loc="upper right", frameon=False, fontsize=7.5)

combined_min = min(X_resc.min(), X_kmeans.min())
combined_max = max(X_resc.max(), X_kmeans.max())
padding = 0.06 * (combined_max - combined_min)
limits = (combined_min - padding, combined_max + padding)

for ax, X_output, y_output, title in [
    (axes[1], X_resc, y_resc, "(b) Re-SC output"),
    (axes[2], X_kmeans, y_kmeans, "(c) KMeans-ReSC output"),
]:
    ax.scatter(
        X_output[y_output == 0, 0], X_output[y_output == 0, 1],
        s=5, c=majority_color, alpha=0.10, linewidths=0, rasterized=True,
        label="Majority",
    )
    ax.scatter(
        X_output[y_output == 1, 0], X_output[y_output == 1, 1],
        s=5, c=minority_color, alpha=0.10, linewidths=0, rasterized=True,
        label="Minority",
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("First component $x_1$")
    ax.set_ylabel("Second component $x_2$")
    ax.set_xlim(limits)
    ax.set_ylim(limits)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.16, linewidth=0.45)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8)

pdf_path = figure_dir / "one_dimensional_resampling_example.pdf"
png_path = figure_dir / "one_dimensional_resampling_example.png"
fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {pdf_path}")
print(f"Saved: {png_path}")

## Suggested paper caption

**Illustrative one-dimensional resampling example.** (a) The original binary dataset contains a separated majority mode and a second majority mode that overlaps the minority distribution. (b) Re-SC may retain majority representatives from the overlap region through its density-weighted stochastic selection. (c) In this fixed example, KMeans-ReSC filters locally ambiguous majority inputs before extracting centroid representatives, reducing visible class overlap after concatenation. The example illustrates algorithmic geometry and is not evidence of universal superiority.